# Aula 4, GRU

Notebook executável que acompanha a aula [04-gru.md](../../lessons/modulo-05-deep-learning-nlp/04-gru.md).

## O que vamos fazer aqui

Duas partes. Primeiro, implementamos a passagem para frente de uma célula GRU, com os
seus dois portões. Depois, no projeto que fecha o módulo, construímos um classificador
neural de intenção de mensagens de alunos, juntando o Bag of Words do Módulo 3 com a
rede neural da primeira aula deste módulo. Há ainda um caminho opcional com uma GRU de
verdade em PyTorch.

## A célula GRU

A GRU tem dois portões, o de reinício e o de atualização, e um único estado. A função
abaixo faz uma passagem da célula e processa uma pequena sequência.

In [ ]:
import numpy as np


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def gru_passo(x, h, P):
    """Uma passagem da célula GRU, com portões de reinício e atualização."""
    z = np.concatenate([x, h])
    r = sigmoide(P["Wr"] @ z + P["br"])             # portão de reinício
    u = sigmoide(P["Wu"] @ z + P["bu"])             # portão de atualização
    candidato = np.tanh(P["Wh"] @ np.concatenate([x, r * h]) + P["bh"])
    return (1 - u) * h + u * candidato              # combina passado e candidato


H, Dx = 5, 1
rng = np.random.default_rng(2)
P = {k: rng.normal(0, 0.3, (H, H + Dx)) for k in ["Wr", "Wu", "Wh"]}
P.update({k: np.zeros(H) for k in ["br", "bu", "bh"]})

h = np.zeros(H)
for x in [0.0, 1.0, 0.0, 1.0]:
    h = gru_passo(np.array([x]), h, P)

print("estado final da GRU após a sequência:")
print(np.round(h, 3))

A célula processa a sequência e produz um estado final, com os dois portões
funcionando. Com menos parâmetros que a LSTM, a GRU chega a um mecanismo de memória
parecido.

## Projeto do módulo, classificador neural de intenção

Vamos classificar mensagens curtas de alunos em três intenções: dúvida, elogio e
problema técnico. Representamos cada mensagem com Bag of Words, removendo stopwords (do
Módulo 3) para destacar os termos de conteúdo, e classificamos com uma rede neural de
uma camada escondida e saída softmax sobre as três classes.

In [ ]:
import re

STOP = {'a', 'o', 'as', 'os', 'um', 'uma', 'de', 'da', 'do', 'em', 'na', 'no', 'que',
        'é', 'e', 'com', 'qual', 'para', 'por', 'não', 'muito', 'você', 'pela',
        'essa', 'está', 'fica', 'fora', 'ar'}


def tokenizar(t):
    return [w for w in re.findall(r"\w+", t.lower()) if w not in STOP]


mensagens = [
    ("não entendi a derivada", "dúvida"),
    ("como resolvo essa integral", "dúvida"),
    ("não consigo entender a matriz", "dúvida"),
    ("qual a fórmula do limite", "dúvida"),
    ("muito obrigado pela aula", "elogio"),
    ("adorei a explicação", "elogio"),
    ("aula excelente parabéns", "elogio"),
    ("você explica muito bem", "elogio"),
    ("o vídeo não carrega", "técnico"),
    ("a página está travando", "técnico"),
    ("não consigo fazer login", "técnico"),
    ("o site fica fora do ar", "técnico"),
]

vocab = sorted({w for m, _ in mensagens for w in tokenizar(m)})
vi = {w: i for i, w in enumerate(vocab)}
classes = ["dúvida", "elogio", "técnico"]
ci = {c: i for i, c in enumerate(classes)}


def bow(texto):
    v = np.zeros(len(vocab))
    for w in tokenizar(texto):
        if w in vi:
            v[vi[w]] += 1
    return v


X = np.array([bow(m) for m, _ in mensagens])
Y = np.zeros((len(mensagens), 3))
for k, (_, c) in enumerate(mensagens):
    Y[k, ci[c]] = 1

Com os dados em vetores, treinamos a rede com saída softmax e custo de
entropia cruzada, o mesmo motor de backpropagation da primeira aula, agora com três
classes.

In [ ]:
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


rng = np.random.default_rng(0)
H = 8
W1 = rng.normal(0, 0.5, (len(vocab), H)); b1 = np.zeros((1, H))
W2 = rng.normal(0, 0.5, (H, 3)); b2 = np.zeros((1, 3))

for _ in range(3000):
    a1 = np.tanh(X @ W1 + b1)
    p = softmax(a1 @ W2 + b2)
    d2 = (p - Y) / len(X)
    d1 = (d2 @ W2.T) * (1 - a1 ** 2)
    W2 -= 0.5 * a1.T @ d2; b2 -= 0.5 * d2.sum(0, keepdims=True)
    W1 -= 0.5 * X.T @ d1; b1 -= 0.5 * d1.sum(0, keepdims=True)


def prever(texto):
    a1 = np.tanh(bow(texto) @ W1 + b1)
    return classes[int(np.argmax(softmax(a1 @ W2 + b2)))]


treino_acc = np.mean([prever(m) == c for m, c in mensagens])
teste = [
    ("não entendi o limite", "dúvida"),
    ("parabéns pela aula", "elogio"),
    ("o login não funciona", "técnico"),
    ("adorei a matéria", "elogio"),
]
print(f"Acurácia no treino: {treino_acc:.0%}\n")
acertos = 0
for m, c in teste:
    p = prever(m)
    acertos += (p == c)
    print(f"  {m!r:30} -> {p:9} (correto: {c})")
print(f"\nAcurácia no teste: {acertos}/{len(teste)}")

A rede acerta 100% no treino e as quatro mensagens de teste, mesmo elas usando
palavras que não estão idênticas no treino, porque os termos de conteúdo apontam a
intenção certa. Note como a remoção de stopwords, lá do Módulo 3, foi importante para o
modelo não se confundir com palavras comuns como não e o.

## Caminho opcional, GRU de verdade com PyTorch

Esta célula treina uma GRU sobre a sequência de palavras, levando em conta a ordem, algo
que o Bag of Words ignora. Ela roda apenas se o PyTorch estiver instalado, e avisa caso
contrário.

In [ ]:
try:
    import torch
    import torch.nn as nn

    # Vocabulário com índice para padding.
    palavra_idx = {w: i + 1 for i, w in enumerate(vocab)}  # 0 reservado para padding
    maxlen = max(len(tokenizar(m)) for m, _ in mensagens)

    def codificar(texto):
        ids = [palavra_idx.get(w, 0) for w in tokenizar(texto)]
        return ids + [0] * (maxlen - len(ids))

    Xt = torch.tensor([codificar(m) for m, _ in mensagens])
    yt = torch.tensor([ci[c] for _, c in mensagens])

    class ClassificadorGRU(nn.Module):
        def __init__(self, vocab_size, emb=16, hidden=16, n_classes=3):
            super().__init__()
            self.emb = nn.Embedding(vocab_size + 1, emb, padding_idx=0)
            self.gru = nn.GRU(emb, hidden, batch_first=True)
            self.fc = nn.Linear(hidden, n_classes)

        def forward(self, x):
            _, h = self.gru(self.emb(x))
            return self.fc(h[-1])

    modelo = ClassificadorGRU(len(vocab))
    opt = torch.optim.Adam(modelo.parameters(), lr=0.05)
    perda = nn.CrossEntropyLoss()
    for _ in range(300):
        opt.zero_grad()
        saida = modelo(Xt)
        erro = perda(saida, yt)
        erro.backward()
        opt.step()
    acc = (modelo(Xt).argmax(1) == yt).float().mean().item()
    print(f"GRU em PyTorch, acurácia no treino: {acc:.0%}")
except Exception as erro:
    print("PyTorch não disponível, ficamos com o classificador Bag of Words acima.")
    print("Para instalar, veja docs/setup.md. Detalhe:", erro)

A GRU lê a sequência de palavras na ordem, o que lhe permite captar padrões que
o Bag of Words perde. Para o projeto do módulo, compare os dois classificadores e discuta
o que a ordem das palavras acrescenta. Com isso, você fecha o módulo de Deep Learning
para NLP e fica pronto para os Transformers do Módulo 6.